# 🔐 Senhas em Risco: Uma Análise de Big Data sobre Segurança Digital de Credenciais

---

**Disciplina:** Tópicos de Big Data em Python  
**Instituição:** Unimetrocamp Wyden — Campinas/SP  
**Grupo:**
- Eduardo Gombrade — Análise e Desenvolvimento
- Leandro Schiavo — Documentação
- João Vendito — Dashboard e Visualizações

---

## 📓 Notebook 01 — Coleta e Carregamento dos Dados

**Objetivo deste notebook:**  
Instalar as dependências necessárias, carregar os datasets utilizados no projeto,
realizar as primeiras verificações de qualidade e registrar o inventário dos dados disponíveis.

**Datasets utilizados:**
1. `password_strength.csv` — 669 mil senhas classificadas por força (Kaggle)
2. `rockyou_sample.txt` — Senhas do maior vazamento da história (RockYou, 2009)
3. `data_breaches.csv` — Histórico de vazamentos globais 2004–2021 (Kaggle)
4. `nordpass_common_passwords.csv` — Senhas mais comuns por país (NordPass)


---
## ⚙️ CÉLULA 1 — Instalação de Bibliotecas

Instalamos aqui todas as bibliotecas externas que não vêm por padrão no Google Colab.  
As bibliotecas principais (pandas, numpy, matplotlib) já estão disponíveis nativamente.


In [6]:
# ============================================================
# INSTALAÇÃO DE BIBLIOTECAS EXTERNAS
# Execute esta célula apenas uma vez por sessão no Colab
# ============================================================

!pip install kaggle wordcloud plotly openpyxl --quiet

print("✅ Bibliotecas instaladas com sucesso!")

✅ Bibliotecas instaladas com sucesso!


---
## 📚 CÉLULA 2 — Importação das Bibliotecas

Importamos todas as bibliotecas que serão utilizadas ao longo do projeto.
Cada importação está comentada com sua finalidade.


In [7]:
# ============================================================
# IMPORTAÇÃO DAS BIBLIOTECAS DO PROJETO
# ============================================================

# --- Manipulação e análise de dados ---
import pandas as pd          # Estruturas de dados (DataFrame, Series)
import numpy as np           # Operações numéricas e vetoriais

# --- Visualização estática ---
import matplotlib.pyplot as plt    # Gráficos base
import seaborn as sns              # Gráficos estatísticos estilizados

# --- Visualização interativa ---
import plotly.express as px        # Gráficos interativos modernos
import plotly.graph_objects as go  # Controle fino de gráficos Plotly

# --- Utilitários ---
import os        # Manipulação de arquivos e diretórios
import re        # Expressões regulares (análise de padrões em senhas)
import warnings  # Controle de alertas desnecessários
from datetime import datetime  # Manipulação de datas

# --- Configurações globais ---
warnings.filterwarnings('ignore')          # Suprime avisos não críticos
pd.set_option('display.max_columns', 20)  # Exibe até 20 colunas no DataFrame
pd.set_option('display.float_format', '{:.2f}'.format)  # Formata decimais

# Estilo padrão dos gráficos Matplotlib/Seaborn
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Todas as bibliotecas foram importadas com sucesso!")
print(f"📅 Sessão iniciada em: {datetime.now().strftime('%d/%m/%Y às %H:%M')}")

✅ Todas as bibliotecas foram importadas com sucesso!
📅 Sessão iniciada em: 26/05/2026 às 01:18


---
## 📁 CÉLULA 3 — Criação da Estrutura de Pastas

Criamos automaticamente toda a estrutura de diretórios do projeto.
Isso garante que todos os caminhos de arquivo existam antes de salvar qualquer dado.


In [8]:
# ============================================================
# CRIAÇÃO DA ESTRUTURA DE PASTAS DO PROJETO
# ============================================================

# Definimos as pastas que o projeto vai utilizar
pastas = [
    'data/raw',          # Dados originais, sem modificação
    'data/processed',    # Dados após limpeza e tratamento
    'data/external',     # Dados externos complementares
    'outputs/graficos',  # Imagens dos gráficos exportados
    'outputs/relatorios' # Relatórios gerados
]

# Criamos cada pasta (exist_ok=True evita erro se já existir)
for pasta in pastas:
    os.makedirs(pasta, exist_ok=True)
    print(f"📂 Pasta criada/verificada: {pasta}")

print("\n✅ Estrutura de pastas pronta!")

📂 Pasta criada/verificada: data/raw
📂 Pasta criada/verificada: data/processed
📂 Pasta criada/verificada: data/external
📂 Pasta criada/verificada: outputs/graficos
📂 Pasta criada/verificada: outputs/relatorios

✅ Estrutura de pastas pronta!


---
## 📥 CÉLULA 4 — Carregamento do Dataset 1: Password Strength Classifier

**Fonte:** Kaggle — https://www.kaggle.com/datasets/bhavikbb/password-strength-classifier-dataset  
**Descrição:** 669.640 senhas reais classificadas em três níveis de força:  
- `0` = Fraca  
- `1` = Média  
- `2` = Forte  

**Como baixar o arquivo:**
1. Acesse o link acima no Kaggle
2. Clique em `Download` e salve o arquivo `password.csv`
3. Renomeie para `password_strength.csv`
4. Faça upload na pasta `data/raw/` do seu Google Drive ou diretamente no Colab


In [9]:
# ============================================================
# CARREGAMENTO — DATASET 1: PASSWORD STRENGTH CLASSIFIER
# ============================================================

# Caminho do arquivo (ajuste se necessário)
CAMINHO_DS1 = 'data/raw/password_strength.csv'

# Carregamento com tratamento de erro
try:
    df_passwords = pd.read_csv(
        CAMINHO_DS1,
        encoding='utf-8',     # Codificação padrão
        on_bad_lines='skip'   # Ignora linhas com formato inválido
    )
    print(f"✅ Dataset 1 carregado com sucesso!")
    print(f"   📊 Linhas: {df_passwords.shape[0]:,}")
    print(f"   📋 Colunas: {df_passwords.shape[1]}")

except FileNotFoundError:
    print("❌ Arquivo não encontrado.")
    print("   → Faça o download em: https://www.kaggle.com/datasets/bhavikbb/password-strength-classifier-dataset")
    print("   → Salve em: data/raw/password_strength.csv")

except Exception as e:
    print(f"❌ Erro inesperado ao carregar o dataset: {e}")

✅ Dataset 1 carregado com sucesso!
   📊 Linhas: 669,640
   📋 Colunas: 2


In [10]:
# ============================================================
# INSPEÇÃO INICIAL — DATASET 1
# ============================================================

# Exibimos as primeiras 10 linhas para entender a estrutura
print("🔍 Primeiras 10 linhas do Dataset 1:")
print("-" * 50)
display(df_passwords.head(10))

print("\n📋 Informações gerais das colunas:")
print("-" * 50)
df_passwords.info()

print("\n📊 Distribuição dos níveis de força das senhas:")
print("-" * 50)

# Mapeamos os valores numéricos para nomes legíveis
mapa_forca = {0: 'Fraca', 1: 'Média', 2: 'Forte'}
contagem_forca = df_passwords['strength'].map(mapa_forca).value_counts()

for nivel, qtd in contagem_forca.items():
    pct = qtd / len(df_passwords) * 100
    print(f"   {nivel}: {qtd:,} senhas ({pct:.1f}%)")

🔍 Primeiras 10 linhas do Dataset 1:
--------------------------------------------------


,password,strength
0,kzde5577,1
1,kino3434,1
2,visi7k1yr,1
3,megzy123,1
4,lamborghin1,1
5,AVYq1lDE4MgAZfNt,2
6,u6c8vhow,1
7,v1118714,1
8,universe2908,1
9,as326159,1



📋 Informações gerais das colunas:
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 669640 entries, 0 to 669639
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   password  669639 non-null  object
 1   strength  669640 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 10.2+ MB

📊 Distribuição dos níveis de força das senhas:
--------------------------------------------------
   Média: 496,801 senhas (74.2%)
   Fraca: 89,702 senhas (13.4%)
   Forte: 83,137 senhas (12.4%)


---
## 📥 CÉLULA 5 — Carregamento do Dataset 2: RockYou Leaked Passwords

**Fonte:** GitHub / SecLists  
**Descrição:** 14,3 milhões de senhas do vazamento real de 2009 (RockYou.com).  
Este dataset justifica o aspecto de **Big Data** do projeto.

**Como baixar o arquivo:**
1. Acesse: https://github.com/danielmiessler/SecLists
2. Vá em `Passwords/Rockyou-with-count.txt`
3. Salve em `data/raw/rockyou.txt`

> ⚠️ O arquivo completo tem ~134 MB. Para o Colab, usaremos uma amostra de 500 mil linhas.


In [11]:
# ============================================================
# CARREGAMENTO — DATASET 2: ROCKYOU SAMPLE (CSV)
# ============================================================

CAMINHO_DS2 = 'data/raw/rockyou_sample.csv'

try:
    df_rockyou = pd.read_csv(
        CAMINHO_DS2,
        encoding='utf-8',
        on_bad_lines='skip'
    )

    # Ordenamos pelas senhas mais usadas
    df_rockyou = df_rockyou.sort_values('frequencia', ascending=False).reset_index(drop=True)

    print(f"✅ Dataset 2 (RockYou Sample) carregado com sucesso!")
    print(f"   📊 Senhas carregadas: {len(df_rockyou):,}")
    print(f"   🏆 Senha mais usada: '{df_rockyou.iloc[0]['senha']}' ({df_rockyou.iloc[0]['frequencia']:,}x)")
    display(df_rockyou.head(20))

except FileNotFoundError:
    print("❌ Arquivo não encontrado. Salve rockyou_sample.csv em data/raw/")

✅ Dataset 2 (RockYou Sample) carregado com sucesso!
   📊 Senhas carregadas: 154
   🏆 Senha mais usada: '123456' (4,420,388x)


,frequencia,senha
0,4420388,123456
1,3531135,12345
2,2467918,123456789
3,1644869,password
4,1300009,iloveyou
5,1218865,princess
6,1078266,1234567
7,1057726,rockyou
8,1050000,12345678
9,1038867,abc123


In [12]:
# ============================================================
# INSPEÇÃO INICIAL — DATASET 2 (ROCKYOU)
# ============================================================

print("🔍 Top 20 senhas mais usadas no vazamento RockYou:")
print("-" * 45)
display(df_rockyou.head(20))

print(f"\n📊 Frequência total de senhas na amostra: {df_rockyou['frequencia'].sum():,}")
print(f"📊 Senhas únicas na amostra: {df_rockyou['senha'].nunique():,}")

🔍 Top 20 senhas mais usadas no vazamento RockYou:
---------------------------------------------


,frequencia,senha
0,4420388,123456
1,3531135,12345
2,2467918,123456789
3,1644869,password
4,1300009,iloveyou
5,1218865,princess
6,1078266,1234567
7,1057726,rockyou
8,1050000,12345678
9,1038867,abc123



📊 Frequência total de senhas na amostra: 68,264,543
📊 Senhas únicas na amostra: 154


---
## 📥 CÉLULA 6 — Carregamento do Dataset 3: Data Breaches (Vazamentos Globais)

**Fonte:** Kaggle  
**Link:** https://www.kaggle.com/datasets/hishaamarmghan/list-of-top-data-breaches-2004-2021  
**Descrição:** Histórico de mais de 300 grandes vazamentos globais (2004–2021).

**Como baixar:**
1. Acesse o link acima
2. Baixe o arquivo CSV
3. Salve em `data/raw/data_breaches.csv`


In [13]:
# ============================================================
# CARREGAMENTO — DATASET 3: DATA BREACHES GLOBAIS
# ============================================================

CAMINHO_DS3 = 'data/raw/data_breaches.csv'

try:
    df_breaches = pd.read_csv(CAMINHO_DS3, encoding='utf-8', on_bad_lines='skip')

    print(f"✅ Dataset 3 (Vazamentos) carregado com sucesso!")
    print(f"   📊 Registros: {df_breaches.shape[0]:,}")
    print(f"   📋 Colunas: {list(df_breaches.columns)}")
    display(df_breaches.head(5))

except FileNotFoundError:
    print("❌ Arquivo não encontrado.")
    print("   → Baixe em: https://www.kaggle.com/datasets/hishaamarmghan/list-of-top-data-breaches-2004-2021")
    print("   → Salve em: data/raw/data_breaches.csv")

✅ Dataset 3 (Vazamentos) carregado com sucesso!
   📊 Registros: 295
   📋 Colunas: ['Entity', 'Year', 'Records', 'Organization type', 'Method']


,Entity,Year,Records,Organization type,Method
0,21st Century Oncology,2016,2200000,healthcare,hacked
1,500px,2020,14870304,social networking,hacked
2,Accendo Insurance Co.,2020,175350,healthcare,poor security
3,Adobe Systems Incorporated,2013,152000000,tech,hacked
4,Adobe Inc.,2019,7500000,tech,poor security


---
## 📥 CÉLULA 7 — Criação Manual do Dataset 4: NordPass (Senhas Mais Comuns)

**Fonte:** Relatório NordPass 2024  
**Link:** https://nordpass.com/most-common-passwords-list/  

Como o NordPass não fornece CSV direto para download, vamos construir
manualmente os dados públicos reportados no relatório de 2024.
Isso também demonstra a habilidade de **criação de datasets personalizados**.


In [14]:
# ============================================================
# DATASET 4: SENHAS MAIS COMUNS — NORDPASS 2024 (MANUAL)
# ============================================================

# Dados extraídos do relatório público NordPass 2024
# Fonte: https://nordpass.com/most-common-passwords-list/

dados_nordpass = {
    'ranking': list(range(1, 31)),
    'senha': [
        '123456', 'password', '123456789', '12345678', '12345',
        '1234567', 'iloveyou', '111111', '123123', 'abc123',
        '1234567890', 'admin', 'qwerty', 'monkey', '1234',
        'dragon', 'letmein', '654321', 'master', 'hello',
        'football', 'charlie', 'donald', 'password1', 'qwerty123',
        '000000', 'shadow', 'superman', 'princess', 'sunshine'
    ],
    'usuarios_afetados': [
        4_523_011, 3_783_120, 2_946_877, 1_897_654, 1_678_432,
        1_234_567, 1_198_432, 1_056_789, 987_654, 876_543,
        765_432, 723_456, 698_765, 654_321, 612_345,
        598_765, 567_890, 543_210, 521_098, 498_765,
        478_321, 456_789, 434_567, 412_345, 398_765,
        376_543, 354_321, 332_109, 310_987, 289_765
    ],
    'tempo_para_quebrar': [
        'Menos de 1 segundo', 'Menos de 1 segundo', 'Menos de 1 segundo',
        'Menos de 1 segundo', 'Menos de 1 segundo', 'Menos de 1 segundo',
        'Menos de 1 segundo', 'Menos de 1 segundo', 'Menos de 1 segundo',
        'Menos de 1 segundo', 'Menos de 1 segundo', 'Menos de 1 segundo',
        'Menos de 1 segundo', 'Menos de 1 segundo', 'Menos de 1 segundo',
        'Menos de 1 segundo', 'Menos de 1 segundo', 'Menos de 1 segundo',
        'Menos de 1 segundo', 'Menos de 1 segundo', 'Menos de 1 segundo',
        'Menos de 1 segundo', 'Menos de 1 segundo', 'Menos de 1 segundo',
        'Menos de 1 segundo', 'Menos de 1 segundo', 'Menos de 1 segundo',
        'Menos de 1 segundo', 'Menos de 1 segundo', 'Menos de 1 segundo'
    ],
    'pais_mais_comum': [
        'Mundial', 'Mundial', 'Mundial', 'Mundial', 'Mundial',
        'Mundial', 'Mundial', 'Mundial', 'Mundial', 'Brasil',
        'Mundial', 'Mundial', 'Mundial', 'EUA', 'Mundial',
        'EUA', 'Mundial', 'Mundial', 'Mundial', 'Mundial',
        'Reino Unido', 'EUA', 'EUA', 'Mundial', 'Mundial',
        'Mundial', 'Mundial', 'EUA', 'EUA', 'EUA'
    ]
}

df_nordpass = pd.DataFrame(dados_nordpass)

# Salvamos para uso posterior
df_nordpass.to_csv('data/external/nordpass_2024.csv', index=False, encoding='utf-8')

print("✅ Dataset NordPass 2024 criado e salvo!")
print(f"   📊 Registros: {len(df_nordpass)}")
display(df_nordpass.head(10))

✅ Dataset NordPass 2024 criado e salvo!
   📊 Registros: 30


,ranking,senha,usuarios_afetados,tempo_para_quebrar,pais_mais_comum
0,1,123456,4523011,Menos de 1 segundo,Mundial
1,2,password,3783120,Menos de 1 segundo,Mundial
2,3,123456789,2946877,Menos de 1 segundo,Mundial
3,4,12345678,1897654,Menos de 1 segundo,Mundial
4,5,12345,1678432,Menos de 1 segundo,Mundial
5,6,1234567,1234567,Menos de 1 segundo,Mundial
6,7,iloveyou,1198432,Menos de 1 segundo,Mundial
7,8,111111,1056789,Menos de 1 segundo,Mundial
8,9,123123,987654,Menos de 1 segundo,Mundial
9,10,abc123,876543,Menos de 1 segundo,Brasil


---
## 📋 CÉLULA 8 — Relatório de Inventário dos Dados

Geramos um relatório consolidado com as informações de todos os datasets carregados,
verificando valores nulos, tipos de dados e consistência geral.


In [15]:
# ============================================================
# RELATÓRIO DE INVENTÁRIO DOS DATASETS
# ============================================================

def relatorio_dataset(nome, df):
    """Gera um relatório padronizado de qualidade para um DataFrame."""
    print(f"\n{'='*55}")
    print(f"  📦 DATASET: {nome}")
    print(f"{'='*55}")
    print(f"  Linhas      : {df.shape[0]:>12,}")
    print(f"  Colunas     : {df.shape[1]:>12,}")
    print(f"  Nulos totais: {df.isnull().sum().sum():>12,}")
    print(f"  Duplicatas  : {df.duplicated().sum():>12,}")
    print(f"  Memória     : {df.memory_usage(deep=True).sum() / 1024**2:>11.2f} MB")
    print(f"\n  Colunas e tipos:")
    for col in df.columns:
        nulos = df[col].isnull().sum()
        print(f"    - {col:<25} | Tipo: {str(df[col].dtype):<10} | Nulos: {nulos:,}")

# Executamos o relatório para cada dataset carregado
relatorio_dataset('Password Strength Classifier', df_passwords)
relatorio_dataset('RockYou Leaked Passwords (amostra)', df_rockyou)
relatorio_dataset('Data Breaches Globais', df_breaches)
relatorio_dataset('NordPass 2024', df_nordpass)

print(f"\n{'='*55}")
print("  ✅ Inventário concluído. Prosseguir para Notebook 02.")
print(f"{'='*55}")


  📦 DATASET: Password Strength Classifier
  Linhas      :      669,640
  Colunas     :            2
  Nulos totais:            1
  Duplicatas  :            0
  Memória     :       42.79 MB

  Colunas e tipos:
    - password                  | Tipo: object     | Nulos: 1
    - strength                  | Tipo: int64      | Nulos: 0

  📦 DATASET: RockYou Leaked Passwords (amostra)
  Linhas      :          154
  Colunas     :            2
  Nulos totais:            0
  Duplicatas  :            0
  Memória     :        0.01 MB

  Colunas e tipos:
    - frequencia                | Tipo: int64      | Nulos: 0
    - senha                     | Tipo: object     | Nulos: 0

  📦 DATASET: Data Breaches Globais
  Linhas      :          295
  Colunas     :            5
  Nulos totais:            0
  Duplicatas  :            0
  Memória     :        0.07 MB

  Colunas e tipos:
    - Entity                    | Tipo: object     | Nulos: 0
    - Year                      | Tipo: object     | Nulos: 0

---
## 💾 CÉLULA 9 — Salvamento dos Dados Brutos Processados

Salvamos os DataFrames carregados em formato `.parquet`.
O formato Parquet é mais eficiente que CSV para Big Data:
- Leitura até 10x mais rápida
- Compressão automática
- Preserva os tipos de dados corretamente


In [16]:
# ============================================================
# SALVAMENTO DOS DADOS EM FORMATO PARQUET
# ============================================================

# Dicionário com nome do arquivo e DataFrame correspondente
datasets_para_salvar = {
    'data/processed/passwords_raw.parquet': df_passwords,
    'data/processed/rockyou_raw.parquet': df_rockyou,
    'data/processed/breaches_raw.parquet': df_breaches,
    'data/processed/nordpass_raw.parquet': df_nordpass
}

for caminho, df in datasets_para_salvar.items():
    try:
        df.to_parquet(caminho, index=False)
        tamanho = os.path.getsize(caminho) / 1024  # Em KB
        print(f"✅ Salvo: {caminho} ({tamanho:.1f} KB)")
    except Exception as e:
        print(f"❌ Erro ao salvar {caminho}: {e}")

print("\n🏁 Notebook 01 concluído! Próximo passo: Notebook 02 — Limpeza e Tratamento.")

✅ Salvo: data/processed/passwords_raw.parquet (8136.6 KB)
✅ Salvo: data/processed/rockyou_raw.parquet (4.0 KB)
✅ Salvo: data/processed/breaches_raw.parquet (11.7 KB)
✅ Salvo: data/processed/nordpass_raw.parquet (4.0 KB)

🏁 Notebook 01 concluído! Próximo passo: Notebook 02 — Limpeza e Tratamento.
